In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

# libraries needed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

# reading and loading CSV file

delivery_path = os.path.join(path, 'Q1_data.csv')
df_delivery = pd.read_csv(delivery_path)


In [ ]:
# Task 2: Write your code here:

# inspecting shape and head

print(f"Shape: {df_delivery.shape}")
df_delivery.head()

In [ ]:
# Task 3: Write your code here:

# info
df_delivery.info()

# this mean delivery time has some null values

In [ ]:
# Task 4: Write your code here:

# describe

df_delivery.describe()

In [ ]:
# Task 5: Write your code here:

# bar plot for delivery time which is the target

plt.figure(figsize=(10, 5))
plt.hist(df_delivery['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Time in Minutes')
plt.ylabel('Frequency')
plt.show()

# few big outliers at 140+

In [ ]:
# Task 1: Write your code here:

# dropping order id column
df_clean = df_delivery.drop(columns=['Order_ID'])
print(f"Shape after cleaning: {df_clean.shape}")

df_clean.head()

In [ ]:
# Task 2: Write your code here:

# Analyzing for missing values
missing_percentage = (df_delivery.isnull().sum() / len(df_delivery)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
print(missing_data.head(10))
print('-------------------------------------------------------')

# missing values are very minimal as shown below

# we can solve them all by dropping rows with missing values as they are very minimal damage, I'd rather have clean accurate data then filling with mean or mode, we will see how ti works out

empty_val_col = ['Delivery_Time', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']



print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=empty_val_col)
print(f"After dropping missing vals: {df_clean.shape}")

#removed rows with missing values

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:


categorical_cols = df_clean.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))


label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean.head(10)


In [ ]:
# Task 5: Write your code here:

# scaling using standard


scaler = StandardScaler()

df_scaled_data = pd.DataFrame(scaler.fit_transform(df_clean))

df_scaled_data


In [ ]:
# Task 6: Write your code here:

# target imbalance


df_clean.describe()

In [ ]:
# Task 1: Write your code here:

X = df_scaled_data.drop(7, axis=1)
y = df_scaled_data.drop(7)


In [ ]:
# Task 2,3,4,5: Write your code here:

print("Splitting data into training and testing sets...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split successful.")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("Initializing models and K-Fold cross-validation...")

# Determine the number of classes for one-hot encoding
num_classes = len(np.unique(y_encoded))

# 1. Initialize the machine learning models
models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', max_iter=200, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42) # probability=True is needed for .predict_proba
}

# 2. Create a KFold object
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Create an empty dictionary to store average losses
model_losses = {}

print("Starting K-Fold Cross-Validation for each model...")

# 4. For each model:
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    fold_losses = [] # List to store loss from each fold

    for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
        # Split X_train and y_train into training and validation sets for the current fold
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        # Train the current model
        model.fit(X_train_fold, y_train_fold)

        # Generate predictions (probabilities) on the validation set
        y_pred_proba = model.predict_proba(X_val_fold)

        # Convert y_val_fold (true labels) into a one-hot encoded format
        y_val_one_hot = one_hot_encode(y_val_fold, num_classes)

        # Calculate categorical cross-entropy loss for the current fold
        loss = categorical_cross_entropy(y_val_one_hot, y_pred_proba)
        fold_losses.append(loss)
print("Initializing models and K-Fold cross-validation...")

# Determine the number of classes for one-hot encoding
num_classes = len(np.unique(y_encoded))

# 1. Initialize the machine learning models
models = {
    'Logistic Regression': LogisticRegression(solver='liblinear', max_iter=200, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42) # probability=True is needed for .predict_proba
}

# 2. Create a KFold object
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Create an empty dictionary to store average losses
model_losses = {}

print("Starting K-Fold Cross-Validation for each model...")

# 4. For each model:
for model_name, model in models.items():
    print(f"\nTraining and evaluating {model_name}...")
    fold_losses = [] # List to store loss from each fold

    for fold, (train_index, val_index) in enumerate(kf.split(X_train)):
        # Split X_train and y_train into training and validation sets for the current fold
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        # Train the current model
        model.fit(X_train_fold, y_train_fold)

        # Generate predictions (probabilities) on the validation set
        y_pred_proba = model.predict_proba(X_val_fold)

        # Convert y_val_fold (true labels) into a one-hot encoded format
        y_val_one_hot = one_hot_encode(y_val_fold, num_classes)

        # Calculate categorical cross-entropy loss for the current fold
        loss = categorical_cross_entropy(y_val_one_hot, y_pred_proba)
        fold_losses.append(loss)

    # Calculate the average loss for the model
    avg_loss = np.mean(fold_losses)
    model_losses[model_name] = avg_loss

    # Print the average cross-validation loss for the current model
    print(f"{model_name} - Average Cross-Validation Loss: {avg_loss:.4f}")

print("\nAll models trained and evaluated. Stored average losses:")
print(model_losses)






In [ ]:
# Task 1: Write your code here:
#TODO: Calculate the average losses across folds
avg_loss = np.mean(sr_results['loss'], axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Training', color='purple')
plt.title('Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('Categorical Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 2: Write your code here:
#TODO: Calculate the average losses across folds
avg_loss = np.mean(sr_results['loss'], axis=0)

plt.figure(figsize=(10, 6))
plt.plot(avg_loss, label='Training', color='purple')
plt.title('Loss Curve')
plt.xlabel('Iteration')
plt.ylabel('Categorical Cross-Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Task Bonus: Write your code here: